# Derm7pt - Classification head com MedSigLIP + Focal Loss
Usa o vision encoder do MedGemma 4B (MedSigLIP) como feature extractor congelado, combinado com metadados demograficos via classification head. Treino com Focal Loss + class weights.

In [ ]:
import os
import sys
import site
import importlib
import subprocess

REPO_URL = "https://github.com/RodrigoAraujo12/melanoma-tcc.git"
REPO_DIR = "/kaggle/working/melanoma-tcc"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "transformers", "accelerate", "huggingface_hub"], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()
site.main()

print("Setup OK")

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from collections import Counter
from kaggle_secrets import UserSecretsClient

from melanoma_tcc.data.preprocessing import (
    Derm7ptClassificationDataset, classification_collate_fn,
    GROUP_TO_LABEL, LABEL_TO_GROUP, METADATA_DIM,
)
from melanoma_tcc.model.classifier import build_dermclassifier
from melanoma_tcc.model.losses import FocalLoss, compute_class_weights
from melanoma_tcc.utils.metrics import compute_metrics, plot_confusion_matrix

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret('mellanoma_TCC')

DERM7PT_DIR = "/kaggle/input/datasets/rodrigoadesouza/derm7pt-multimodal/release_v0"
META_CSV = f"{DERM7PT_DIR}/meta/meta.csv"
IMAGES_DIR = f"{DERM7PT_DIR}/images"
TRAIN_IDX = f"{DERM7PT_DIR}/meta/train_indexes.csv"
VAL_IDX = f"{DERM7PT_DIR}/meta/valid_indexes.csv"
TEST_IDX = f"{DERM7PT_DIR}/meta/test_indexes.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Meta CSV existe: {os.path.exists(META_CSV)}")
print(f"Metadata dim: {METADATA_DIM}")

In [ ]:
model, processor = build_dermclassifier(
    hf_token=HF_TOKEN,
    num_classes=5,
    metadata_dim=METADATA_DIM,
    freeze_vision=True,
)
model = model.to(device)
model.vision_encoder = model.vision_encoder.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"Vision hidden size: {model.vision_encoder.config.hidden_size}")

In [ ]:
train_dataset = Derm7ptClassificationDataset(
    META_CSV, IMAGES_DIR, processor,
    indexes_csv=TRAIN_IDX,
    balance=True,
    target_per_class=80,
    max_oversample=4,
    augment=True,
    seed=42,
)
val_dataset = Derm7ptClassificationDataset(
    META_CSV, IMAGES_DIR, processor,
    indexes_csv=VAL_IDX,
    balance=False,
    augment=False,
)
test_dataset = Derm7ptClassificationDataset(
    META_CSV, IMAGES_DIR, processor,
    indexes_csv=TEST_IDX,
    balance=False,
    augment=False,
)

print(f"Train (balanced): {len(train_dataset)}")
print(f"Val: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

train_labels = [train_dataset[i]['labels'].item() for i in range(len(train_dataset))]
print(f"\nTrain distribuicao: {Counter([LABEL_TO_GROUP[l] for l in train_labels])}")
test_labels = [test_dataset[i]['labels'].item() for i in range(len(test_dataset))]
print(f"Test distribuicao: {Counter([LABEL_TO_GROUP[l] for l in test_labels])}")

In [ ]:
BATCH_SIZE = 16
EPOCHS = 15
LR = 5e-4
FOCAL_GAMMA = 2.0
LABEL_SMOOTHING = 0.05

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=classification_collate_fn, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=classification_collate_fn, num_workers=2, pin_memory=True,
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=classification_collate_fn, num_workers=2, pin_memory=True,
)

label_counter = Counter(train_labels)
class_counts = [label_counter.get(i, 1) for i in range(5)]
alpha = compute_class_weights(class_counts, mode="inverse_sqrt")
print(f"Class weights (alpha): {alpha.tolist()}")

criterion = FocalLoss(alpha=alpha, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)
optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            pv = batch['pixel_values'].to(device)
            md = batch['metadata'].to(device)
            lb = batch['labels'].to(device)
            logits = model(pv, md)
            loss = criterion(logits, lb)
            total_loss += loss.item() * lb.size(0)
            all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
            all_labels.extend(lb.cpu().tolist())
    n = len(loader.dataset)
    avg_loss = total_loss / n
    acc = sum(int(p==l) for p,l in zip(all_preds, all_labels)) / n
    return avg_loss, acc, all_preds, all_labels

best_val_acc = 0.0
best_state = None
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    if hasattr(model, 'vision_encoder'):
        model.vision_encoder.eval()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0
    for batch in train_loader:
        pv = batch['pixel_values'].to(device)
        md = batch['metadata'].to(device)
        lb = batch['labels'].to(device)
        logits = model(pv, md)
        loss = criterion(logits, lb)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        optimizer.step()
        train_loss_sum += loss.item() * lb.size(0)
        train_correct += (logits.argmax(dim=-1) == lb).sum().item()
        train_total += lb.size(0)
    scheduler.step()
    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
    history.append((epoch, train_loss, train_acc, val_loss, val_acc))
    print(f"Epoch {epoch:2d}/{EPOCHS} | train_loss={train_loss:.4f} train_acc={train_acc:.3f} | val_loss={val_loss:.4f} val_acc={val_acc:.3f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if 'vision_encoder' not in k}

print(f"\nBest val_acc: {best_val_acc:.3f}")

In [ ]:
if best_state is not None:
    current = model.state_dict()
    current.update(best_state)
    model.load_state_dict(current, strict=False)
    print("Best model carregado.")

os.makedirs("/kaggle/working/derm-classifier-v3", exist_ok=True)
torch.save(best_state, "/kaggle/working/derm-classifier-v3/classifier_head.pt")
print("Salvou /kaggle/working/derm-classifier-v3/classifier_head.pt")

import json
with open("/kaggle/working/derm-classifier-v3/history.json", "w") as f:
    json.dump(history, f, indent=2)

In [ ]:
test_loss, test_acc, test_preds, test_labels_eval = evaluate(model, test_loader, criterion)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

TARGET_NAMES = ["BCC", "NEV", "MEL", "SK", "MISC"]
results = compute_metrics(test_labels_eval, test_preds, target_names=TARGET_NAMES)
plot_confusion_matrix(test_labels_eval, test_preds,
                     save_path='/kaggle/working/derm-classifier-v3-cm.png',
                     target_names=TARGET_NAMES)

In [ ]:
import pandas as pd

test_df = pd.DataFrame({
    'true_label': test_labels_eval,
    'pred_label': test_preds,
    'true_group': [LABEL_TO_GROUP[l] for l in test_labels_eval],
    'pred_group': [LABEL_TO_GROUP[p] for p in test_preds],
})
test_df.to_csv('/kaggle/working/derm_classifier_v3_predictions.csv', index=False)
print(f"Salvas {len(test_df)} predicoes.")
print(f"\nDistribuicao predicoes: {Counter(test_df['pred_group'])}")
print(f"Distribuicao verdadeira: {Counter(test_df['true_group'])}")